In [1]:
import sys
sys.path.append('../')

from gears import PertData, GEARS

Load data. We use norman as an example.

In [2]:
pert_data = PertData('./data')
pert_data.load(data_name = 'norman')
pert_data.prepare_split(split = 'simulation', seed = 1)
pert_data.get_dataloader(batch_size = 32, test_batch_size = 128)

Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'RHOXF2BB+ZBTB25' 'RHOXF2BB+SET']
Local copy of pyg dataset is detected. Loading...
Done!
Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:9
combo_seen1:43
combo_seen2:19
unseen_single:36
Done!
Creating dataloaders....
Done!


here1


In [3]:
# See all unique conditions in the loaded data
print("All conditions in adata:")
print(pert_data.adata.obs['condition'].unique())

# See what the code identifies as single perts in the data
# Filter for conditions that contain 'ctrl' but are not exactly 'ctrl'
all_single_perts_in_data = pert_data.adata.obs[pert_data.adata.obs['condition'].str.contains('ctrl', regex=False) & (pert_data.adata.obs['condition'] != 'ctrl')]['condition'].unique()
print("\nSingle perts identified in adata:")
print(all_single_perts_in_data)
print(f"(Count: {len(all_single_perts_in_data)})")



All conditions in adata:
['TSC22D1+ctrl', 'KLF1+MAP2K6', 'ctrl', 'CEBPE+RUNX1T1', 'MAML2+ctrl', ..., 'STIL+ctrl', 'CDKN1C+ctrl', 'ctrl+CDKN1B', 'CDKN1B+CDKN1A', 'C3orf72+FOXL2']
Length: 277
Categories (277, object): ['AHR+FEV', 'AHR+KLF1', 'AHR+ctrl', 'ARID1A+ctrl', ..., 'ZC3HAV1+HOXC13', 'ZC3HAV1+ctrl', 'ZNF318+FOXL2', 'ZNF318+ctrl']

Single perts identified in adata:
['TSC22D1+ctrl', 'MAML2+ctrl', 'ctrl+CEBPE', 'DUSP9+ctrl', 'ctrl+ELMSAN1', ..., 'SNAI1+ctrl', 'EGR1+ctrl', 'STIL+ctrl', 'CDKN1C+ctrl', 'ctrl+CDKN1B']
Length: 148
Categories (277, object): ['AHR+FEV', 'AHR+KLF1', 'AHR+ctrl', 'ARID1A+ctrl', ..., 'ZC3HAV1+HOXC13', 'ZC3HAV1+ctrl', 'ZNF318+FOXL2', 'ZNF318+ctrl']
(Count: 148)


In [3]:
pert_data.adata

View of AnnData object with n_obs × n_vars = 89357 × 5045
    obs: 'condition', 'cell_type', 'dose_val', 'control', 'condition_name'
    var: 'gene_name'
    uns: 'non_dropout_gene_idx', 'non_zeros_gene_idx', 'rank_genes_groups_cov_all', 'top_non_dropout_de_20', 'top_non_zero_de_20'
    layers: 'counts'

In [25]:
pert_data.adata.obs['condition']

cell_barcode
AAACCTGAGGCATGTG-1     TSC22D1+ctrl
AAACCTGAGGCCCTTG-1      KLF1+MAP2K6
AAACCTGCACGAAGCA-1             ctrl
AAACCTGCAGACGTAG-1    CEBPE+RUNX1T1
AAACCTGCAGCCTTGG-1       MAML2+ctrl
                          ...      
TTTGTCAGTCAGAATA-8             ctrl
TTTGTCATCAGTACGT-8       FOXA3+ctrl
TTTGTCATCCACTCCA-8       CELF2+ctrl
TTTGTCATCCCAACGG-8      BCORL1+ctrl
TTTGTCATCTGGCGAC-8      MAP4K3+ctrl
Name: condition, Length: 89357, dtype: category
Categories (277, object): ['AHR+FEV', 'AHR+KLF1', 'AHR+ctrl', 'ARID1A+ctrl', ..., 'ZC3HAV1+HOXC13', 'ZC3HAV1+ctrl', 'ZNF318+FOXL2', 'ZNF318+ctrl']

In [8]:
all_single_perts_in_data = pert_data.adata.obs[pert_data.adata.obs['condition'].str.contains('ctrl', regex=False) & (pert_data.adata.obs['condition'] != 'ctrl')]['condition'].unique()
print(f"(Count: {len(all_single_perts_in_data)})")


(Count: 148)


In [24]:
all_single_perts_in_data



['TSC22D1+ctrl', 'KLF1+MAP2K6', 'CEBPE+RUNX1T1', 'MAML2+ctrl', 'ctrl+CEBPE', ..., 'STIL+ctrl', 'CDKN1C+ctrl', 'ctrl+CDKN1B', 'CDKN1B+CDKN1A', 'C3orf72+FOXL2']
Length: 276
Categories (277, object): ['AHR+FEV', 'AHR+KLF1', 'AHR+ctrl', 'ARID1A+ctrl', ..., 'ZC3HAV1+HOXC13', 'ZC3HAV1+ctrl', 'ZNF318+FOXL2', 'ZNF318+ctrl']

In [28]:
len(pert_data.pert_names.tolist())

9853

In [29]:

single_pert_names = [p for p in all_single_perts_in_data if p in pert_data.pert_names.tolist()]
single_pert_names

[]

Create a model object; if you use [wandb](https://wandb.ai), you can easily track model training and evaluation by setting `weight_bias_track` to true, and specify the `proj_name` and `exp_name` that you like.

In [4]:
gears_model = GEARS(pert_data, device = 'cuda:0', 
                        weight_bias_track = False, 
                        proj_name = 'gt_gears_contra_pretaining',
                        exp_name = '2layers')
gears_model.model_initialize(hidden_size = 64)


Found local copy...


In [23]:
import os
import pickle
from tqdm import tqdm
import numpy as np
import torch
import torch.optim as optim
import torch.nn as nn
from torch.optim.lr_scheduler import StepLR
import torch.nn.functional as F
import pandas as pd
pert_data.ctrl_expression = torch.tensor(
            np.mean(pert_data.adata.X[pert_data.adata.obs.condition == 'ctrl'],
                    axis=0)).reshape(-1, ).to('cuda:0')
single_pert_names = pert_data.adata.obs[pert_data.adata.obs['condition'].str.contains('ctrl', regex=False) & (pert_data.adata.obs['condition'] != 'ctrl')]['condition'].unique()
delta_vectors = calculate_perturbation_deltas(pert_data, single_pert_names)

Calculating mean delta expression for single perturbations...


Calculating Deltas: 100%|██████████| 148/148 [00:00<00:00, 204.65it/s]


Finished calculating deltas.


In [18]:
def calculate_perturbation_deltas(pert_data, single_pert_list):
    """Calculates mean delta expression for single gene perturbations."""
    print("Calculating mean delta expression for single perturbations...")
    delta_vectors = {}
    control_mean = pert_data.ctrl_expression.cpu().numpy() # Mean control expression

    pert_adata = pert_data.adata[pert_data.adata.obs['condition'].isin(single_pert_list)]

    for pert in tqdm(single_pert_list, desc="Calculating Deltas"):
        pert_mean = np.mean(pert_adata[pert_adata.obs['condition'] == pert].X.toarray(), axis=0)
        delta_vectors[pert] = pert_mean - control_mean

    print("Finished calculating deltas.")
    return delta_vectors

In [26]:
def calculate_similarity_matrix(delta_vectors_dict, pert_names):
    """Calculates pairwise similarity (Pearson correlation) between delta vectors."""
    print("Calculating similarity matrix...")
    # Ensure order matches pert_names
    delta_matrix = np.array([delta_vectors_dict[p] for p in pert_names])
    # Handle potential NaN variance issues if a delta vector is constant zero
    valid_variance = np.std(delta_matrix, axis=1) > 1e-6
    if not np.all(valid_variance):
        print(f"Warning: {np.sum(~valid_variance)} perturbations have near-zero variance in delta expression. Excluding them from similarity calculation.")
        pert_names = [p for i, p in enumerate(pert_names) if valid_variance[i]]
        delta_matrix = delta_matrix[valid_variance,:]
        if len(pert_names) < 2:
                print("Not enough valid perturbations to calculate similarity. Skipping pre-training.")
                return None, None


    # Calculate Pearson correlation matrix
    # Using np_pearson_cor from utils if available, otherwise np.corrcoef
    try:
        from .utils import np_pearson_cor # Try importing the specific function
        sim_matrix = np_pearson_cor(delta_matrix.T, delta_matrix.T) # gene x pert -> sim between perts
    except ImportError:
        print("np_pearson_cor not found in utils, using np.corrcoef.")
        sim_matrix = np.corrcoef(delta_matrix) # pert x pert -> sim between perts

    # Ensure it's symmetric and NaNs are handled (e.g., set to 0)
    sim_matrix = np.nan_to_num(sim_matrix)
    sim_df = pd.DataFrame(sim_matrix, index=pert_names, columns=pert_names)
    print("Finished calculating similarity matrix.")
    return sim_df, pert_names # Return potentially filtered pert_names

In [27]:
sim_df, valid_pert_names = calculate_similarity_matrix(delta_vectors, single_pert_names)

Calculating similarity matrix...
np_pearson_cor not found in utils, using np.corrcoef.
Finished calculating similarity matrix.


In [28]:
sim_df

,TSC22D1+ctrl,MAML2+ctrl,ctrl+CEBPE,DUSP9+ctrl,ctrl+ELMSAN1,ctrl+FOXA1,BCORL1+ctrl,MEIS1+ctrl,GLB1L2+ctrl,KLF1+ctrl,...,ctrl+OSR2,ctrl+FOXL2,ctrl+TGFBR2,KIF18B+ctrl,SPI1+ctrl,SNAI1+ctrl,EGR1+ctrl,STIL+ctrl,CDKN1C+ctrl,ctrl+CDKN1B
TSC22D1+ctrl,1.000000,-0.176886,0.184996,0.288110,-0.150096,0.246345,0.154668,0.355394,0.282727,-0.008987,...,0.143219,0.318842,0.096197,0.363150,0.318480,0.102450,0.408185,0.225472,0.401954,0.367673
MAML2+ctrl,-0.176886,1.000000,-0.300026,0.055497,0.297194,-0.094581,-0.063446,-0.132071,-0.240264,0.195114,...,-0.223698,-0.383726,0.294677,-0.206277,-0.243842,-0.177268,-0.231619,-0.134932,-0.087152,-0.027264
ctrl+CEBPE,0.184996,-0.300026,1.000000,-0.102744,-0.298918,0.152860,0.155529,0.110879,0.245869,-0.218369,...,0.501301,0.516415,-0.257436,0.109176,0.663435,0.371148,0.195715,0.316128,0.210289,0.147576
DUSP9+ctrl,0.288110,0.055497,-0.102744,1.000000,0.101708,0.533861,0.171148,0.577437,0.286470,0.290843,...,-0.154443,0.214228,0.527878,0.552688,0.238686,0.059493,0.566071,0.334105,0.517465,0.574245
ctrl+ELMSAN1,-0.150096,0.297194,-0.298918,0.101708,1.000000,-0.067079,-0.036443,-0.132376,-0.095558,0.401935,...,-0.387764,-0.407311,0.316844,0.051927,-0.130102,-0.388646,-0.013225,-0.050080,0.098104,0.063614
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SNAI1+ctrl,0.102450,-0.177268,0.371148,0.059493,-0.388646,0.365941,0.200917,0.223438,0.267540,-0.143875,...,0.410597,0.579605,-0.013982,0.164406,0.337820,1.000000,0.207413,0.432285,0.240353,0.240292
EGR1+ctrl,0.408185,-0.231619,0.195715,0.566071,-0.013225,0.526875,0.199017,0.569471,0.415369,0.191885,...,0.038433,0.374661,0.258417,0.585484,0.518896,0.207413,1.000000,0.393401,0.595692,0.610773
STIL+ctrl,0.225472,-0.134932,0.316128,0.334105,-0.050080,0.449035,0.268564,0.369104,0.350255,0.045359,...,0.182542,0.483467,0.180769,0.439945,0.509301,0.432285,0.393401,1.000000,0.503152,0.478824
CDKN1C+ctrl,0.401954,-0.087152,0.210289,0.517465,0.098104,0.460589,0.147493,0.459260,0.460395,0.203886,...,-0.024046,0.363809,0.368888,0.553119,0.499371,0.240353,0.595692,0.503152,1.000000,0.853989


In [52]:
single_pert_names

['TSC22D1+ctrl', 'MAML2+ctrl', 'ctrl+CEBPE', 'DUSP9+ctrl', 'ctrl+ELMSAN1', ..., 'SNAI1+ctrl', 'EGR1+ctrl', 'STIL+ctrl', 'CDKN1C+ctrl', 'ctrl+CDKN1B']
Length: 148
Categories (277, object): ['AHR+FEV', 'AHR+KLF1', 'AHR+ctrl', 'ARID1A+ctrl', ..., 'ZC3HAV1+HOXC13', 'ZC3HAV1+ctrl', 'ZNF318+FOXL2', 'ZNF318+ctrl']

In [30]:
gene_list = pert_data.gene_names.values.tolist()
gene_dict = {g:i for i,g in enumerate(gene_list)}
pert_list = pert_data.pert_names.tolist()
pert_data.pert2gene = {p: gene_dict[pert] for p, pert in
                          enumerate(pert_list) if pert in gene_list}

In [ ]:
pert_data.pert2gene

In [36]:

# Map valid perturbation gene names to their indices in the main gene embedding layer
# gene_dict = {g:i for i,g in enumerate(self.gene_list)} # Already calculated in __init__? Check.
# Assuming self.pert2gene map is correct from __init__
pert_indices_in_gene_emb = {name: pert_data.pert2gene[pert_idx]
                            for pert_idx, name in enumerate(pert_list)
                            if name in valid_pert_names and name in gene_list} # Check name exists as a gene
if len(pert_indices_in_gene_emb) < 2:
        
        print("Not enough valid perturbations map to gene embeddings. Skipping pre-training.")
        

Not enough valid perturbations map to gene embeddings. Skipping pre-training.


In [63]:
print(pure_valid_gene_names)
print(len(pure_valid_gene_names))

{'CDKN1C', 'COL2A1', 'MAP2K3', 'CSRNP1', 'MAPK1', 'ELMSAN1', 'CNNM4', 'FOXO4', 'PRTG', 'STIL', 'NIT1', 'MAP7D1', 'RREB1', 'TSC22D1', 'MAP4K3', 'SLC38A2', 'ARRDC3', 'HES7', 'LYL1', 'OSR2', 'HOXC13', 'RUNX1T1', 'FEV', 'SPI1', 'MAP4K5', 'ZBTB25', 'JUN', 'UBASH3B', 'LHX1', 'EGR1', 'COL1A1', 'FOXF1', 'CNN1', 'PTPN1', 'PTPN9', 'HK2', 'IRF1', 'CELF2', 'BPGM', 'PLK4', 'ZNF318', 'UBASH3A', 'BAK1', 'KIF18B', 'ARID1A', 'CDKN1B', 'HOXA13', 'HNF4A', 'SGK1', 'CITED1', 'PTPN13', 'BCL2L11', 'CEBPB', 'IKZF3', 'CLDN6', 'KIF2C', 'PTPN12', 'MAP2K6', 'ETS2', 'TBX3', 'DLX2', 'GLB1L2', 'FOXL2', 'SET', 'PRDM1', 'SLC6A9', 'FOXA3', 'C19orf26', 'CKS1B', 'CEBPE', 'CBL', 'HOXB9', 'C3orf72', 'CEBPA', 'SAMD1', 'TBX2', 'S1PR2', 'AHR', 'DUSP9', 'TMSB4X', 'CDKN1A', 'KLF1', 'ZBTB1', 'ZBTB10', 'CBFA2T3', 'MEIS1', 'SLC4A1', 'TGFBR2', 'KMT2A', 'ATL1', 'POU3F2', 'ISL2', 'FOXA1', 'NCL', 'FOSB', 'MAML2', 'IGDCC3', 'ZC3HAV1', 'TP73', 'BCORL1', 'SNAI1', 'MIDN'}
102


In [64]:
print(valid_pert_names)
print(len(valid_pert_names))

['TSC22D1+ctrl', 'MAML2+ctrl', 'ctrl+CEBPE', 'DUSP9+ctrl', 'ctrl+ELMSAN1', ..., 'SNAI1+ctrl', 'EGR1+ctrl', 'STIL+ctrl', 'CDKN1C+ctrl', 'ctrl+CDKN1B']
Length: 148
Categories (277, object): ['AHR+FEV', 'AHR+KLF1', 'AHR+ctrl', 'ARID1A+ctrl', ..., 'ZC3HAV1+HOXC13', 'ZC3HAV1+ctrl', 'ZNF318+FOXL2', 'ZNF318+ctrl']
148


In [62]:
# Initialize an empty dictionary to store the results
pert_indices_in_gene_emb = {}

# --- NEW STEP: Ensure we have pure gene names from valid_pert_names ---
pure_valid_gene_names = {
    name.replace('+ctrl', '').replace('ctrl+', '') # Replace both possible ctrl additions
    for name in valid_pert_names
}
print(f"Extracted {len(pure_valid_gene_names)} pure gene names from valid_pert_names list.")
# --- End NEW STEP ---

# Loop through all perturbations known to the model (self.pert_list)
# Get both the index (pert_idx) and the name (name) of each perturbation
for pert_idx, name in enumerate(pert_list):

    # Check 1: Is this perturbation name in the list of 'valid' single perturbations
    #          (those suitable for similarity calculation)?
    is_valid_for_similarity = name in pure_valid_gene_names
    print(is_valid_for_similarity)

    # Check 2: Is this perturbation name also a gene that has an embedding
    #          in the main gene embedding layer (i.e., is it in self.gene_list)?
    is_gene_with_embedding = name in gene_list
    print(is_gene_with_embedding)

    # Only proceed if BOTH conditions are true
    if is_valid_for_similarity and is_gene_with_embedding:

        # If yes, find the index for this gene's embedding.
        # self.pert2gene maps the index in self.pert_list (pert_idx)
        # to the index in self.gene_list (the embedding index).
        # We assume pert2gene was correctly built in __init__ to only contain
        # mappings for perturbations that are also in gene_list.
        if pert_idx in pert_data.pert2gene: # Add a safety check
            embedding_index = pert_data.pert2gene[pert_idx]

            # Add the entry to our result dictionary:
            # Key = perturbation/gene name
            # Value = index in the embedding layer
            pert_indices_in_gene_emb[name] = embedding_index
        # else: # Optional: Handle cases where mapping might be missing (shouldn't happen ideally)
            # print_sys(f"Warning: No embedding index found for {name} (pert_idx {pert_idx}) in self.pert2gene.")

pert_indices_in_gene_emb

Extracted 102 pure gene names from valid_pert_names list.
False
False
False
False
False
False
False
False
False
True
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
True
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
True
False
False
False
False
False
False
False
False
False
False
False
False
False
True
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
True
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
True
False
False
False
False


{'AHR': 1869,
 'ARID1A': 98,
 'ARRDC3': 1437,
 'ATL1': 3494,
 'BAK1': 1690,
 'BCL2L11': 699,
 'BCORL1': 2192,
 'BPGM': 2023,
 'C19orf26': 4512,
 'C3orf72': 1047,
 'CBFA2T3': 3973,
 'CBL': 2842,
 'CDKN1A': 1699,
 'CDKN1B': 3095,
 'CDKN1C': 2611,
 'CEBPA': 4654,
 'CEBPB': 4463,
 'CEBPE': 3460,
 'CELF2': 2883,
 'CITED1': 2150,
 'CKS1B': 331,
 'CLDN6': 3783,
 'CNN1': 4579,
 'CNNM4': 676,
 'COL1A1': 4202,
 'COL2A1': 3149,
 'CSRNP1': 917,
 'DLX2': 761,
 'DUSP9': 2223,
 'EGR1': 1489,
 'ELMSAN1': 3544,
 'ETS2': 4994,
 'FEV': 838,
 'FOSB': 4735,
 'FOXA1': 3485,
 'FOXA3': 4738,
 'FOXF1': 3965,
 'FOXL2': 1046,
 'FOXO4': 2144,
 'GLB1L2': 2869,
 'HES7': 4035,
 'HK2': 658,
 'HNF4A': 4445,
 'HOXA13': 1880,
 'HOXB9': 4186,
 'HOXC13': 3198,
 'IGDCC3': 3682,
 'IKZF3': 4131,
 'IRF1': 1477,
 'ISL2': 3706,
 'JUN': 188,
 'KIF18B': 4165,
 'KIF2C': 163,
 'KLF1': 4588,
 'KMT2A': 2837,
 'LHX1': 4115,
 'LYL1': 4592,
 'MAML2': 2806,
 'MAP2K3': 4065,
 'MAP2K6': 4259,
 'MAP4K3': 601,
 'MAP4K5': 3493,
 'MAP7D1': 130

In [54]:
print(len(valid_pert_names))
print(len(pert_list))

148
9853


In [5]:
# --- Add this line ---
gears_model.pretrain_embeddings(pretrain_epochs=15, pretrain_lr=5e-4, temperature=0.07, num_negatives=128, positive_threshold=0.3) # Adjust params as needed
# --------------------

Starting contrastive pre-training of gene embeddings...
Calculating mean delta expression for single perturbations...


(Count: 148)


Calculating Deltas: 100%|██████████| 148/148 [00:01<00:00, 77.52it/s]
Finished calculating deltas.
Calculating similarity matrix...
Finished calculating similarity matrix.
Not enough valid perturbations map to gene embeddings. Skipping pre-training.


You can find available tunable parameters in model_initialize via

In [4]:
gears_model.tunable_parameters()

{'hidden_size': 'hidden dimension, default 64',
 'num_go_gnn_layers': 'number of GNN layers for GO graph, default 1',
 'num_gene_gnn_layers': 'number of GNN layers for co-expression gene graph, default 1',
 'decoder_hidden_size': 'hidden dimension for gene-specific decoder, default 16',
 'num_similar_genes_go_graph': 'number of maximum similar K genes in the GO graph, default 20',
 'num_similar_genes_co_express_graph': 'number of maximum similar K genes in the co expression graph, default 20',
 'coexpress_threshold': 'pearson correlation threshold when constructing coexpression graph, default 0.4',
 'uncertainty': 'whether or not to turn on uncertainty mode, default False',
 'uncertainty_reg': 'regularization term to balance uncertainty loss and prediction loss, default 1',
 'direction_lambda': 'regularization term to balance direction loss and prediction loss, default 1'}

Train your model:

Note: For the sake of demo, we set epoch size to 1. To get full model, set `epochs = 20`.

In [6]:
gears_model.train(epochs = 20,lr= 1e-3)

Start Training...
Epoch 1 Step 1 Train Loss: 0.4685
Epoch 1 Step 51 Train Loss: 0.4960
Epoch 1 Step 101 Train Loss: 0.4475
Epoch 1 Step 151 Train Loss: 0.5446
Epoch 1 Step 201 Train Loss: 0.5705
Epoch 1 Step 251 Train Loss: 0.4541
Epoch 1 Step 301 Train Loss: 0.5437
Epoch 1 Step 351 Train Loss: 0.4914
Epoch 1 Step 401 Train Loss: 0.4212
Epoch 1 Step 451 Train Loss: 0.4950
Epoch 1 Step 501 Train Loss: 0.4824
Epoch 1 Step 551 Train Loss: 0.4925
Epoch 1 Step 601 Train Loss: 0.4863
Epoch 1 Step 651 Train Loss: 0.5178
Epoch 1 Step 701 Train Loss: 0.4316
Epoch 1 Step 751 Train Loss: 0.5290
Epoch 1 Step 801 Train Loss: 0.4720
Epoch 1 Step 851 Train Loss: 0.5581
Epoch 1 Step 901 Train Loss: 0.5769
Epoch 1 Step 951 Train Loss: 0.4238
Epoch 1 Step 1001 Train Loss: 0.4209
Epoch 1 Step 1051 Train Loss: 0.4722
Epoch 1 Step 1101 Train Loss: 0.4301
Epoch 1 Step 1151 Train Loss: 0.4887
Epoch 1 Step 1201 Train Loss: 0.4883
Epoch 1 Step 1251 Train Loss: 0.4383
Epoch 1 Step 1301 Train Loss: 0.5047
Epoch 

Save and load pretrained models:

In [6]:
gears_model.save_model('test_model')
gears_model.load_pretrained('test_model')

Make prediction for new perturbation:

In [7]:
gears_model.predict([['FEV'], ['FEV', 'AHR']])

{'FEV': array([-1.5115363e-06,  4.4304952e-02,  1.0309354e-01, ...,
         3.3967001e+00,  7.8529231e-03,  1.0920237e-31], dtype=float32),
 'FEV_SAMD11': array([-2.2916190e-06,  9.7577907e-02,  1.6493453e-01, ...,
         3.2082996e+00,  7.6769367e-03,  1.7619579e-31], dtype=float32)}

Gene list can be found here:

In [8]:
gears_model.gene_list[:5]

['RP11-34P13.8', 'RP11-54O7.3', 'SAMD11', 'PERM1', 'HES4']